[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_86_Cross_Encoder_Reranking.ipynb)

# Lesson 86 — Cross-Encoder Reranking
### Phase 10 · RAG at Production Scale · Lesson 4 of ~7

You are here because hybrid retrieval (L85) reliably drags the right chunk into the
**top-k** — but it lands at rank #5 or #7 just as often as #1, and the LLM only reads the
top few. This lesson fixes *ordering*. We retrieve wide and cheap, then **rerank** the
shortlist with a model that reads *(query, chunk)* together instead of as two separate
vectors. That is the single highest-leverage precision upgrade in a RAG stack.

**The failure mode we attack — FM3:** *the right chunk was retrieved, but ranked too low to be read.*

## Phase 10 roadmap — the RAG track

| # | Lesson | Failure mode it attacks | Status |
|---|--------|-------------------------|--------|
| L83 | RAG production baseline — where naive RAG breaks | pipeline / retrieval framing | ✅ |
| L84 | Chunking strategies — the ceiling is set at index time | FM4: chunk size & boundaries | ✅ |
| L85 | Dense & hybrid retrieval — embeddings + BM25 + RRF | FM2: the semantic gap | ✅ |
| **L86** | **Cross-encoder reranking — reorder the top-N precisely** | **FM3: right chunk retrieved but ranked too low** | **← you are here** |
| L87 | Grounding & citations — force answers from evidence | FM5: hallucination / no provenance | ⏭ next |
| L88 | RAG evaluation — faithfulness, context precision/recall | FM6: "is it actually better?" | ⏳ |
| L89 | Phase-10 capstone — a production RAG service | integrate everything | ⏳ |

**Design (mirrors L84/L85).** One variable at a time. L85 fixed the chunker and varied the
*retriever*. Today we fix the whole first-stage retriever (L85's `hybrid`) and add a **second
stage** — a reranker — on top of it. Everything before the reranker is held constant so any lift
in the scoreboard is attributable to reranking alone.

## §0 — The problem in one sentence

A retriever scores every chunk against the query **independently and cheaply**. That is what lets
it scan thousands of chunks in milliseconds — but "cheap and independent" is exactly why its
*ordering* is coarse. It gets the right chunk into the top-10, but which of those ten sits at #1 is
close to a coin flip.

Two facts about how the LLM consumes this:

- The generator reads only the **top few** chunks you pass it (context is finite and expensive — L22).
- If the answer chunk is at rank #7 and you pass `k=3`, the model never sees it. **Recall@10 was
  perfect and the answer was still wrong.** That is FM3.

The fix is a *two-stage* pipeline, and it mirrors how every mature search system (Google, Bing,
Elastic, every serious RAG service) is built:

> **Stage 1 — retrieve:** cast a wide, cheap net. Get the right chunk *somewhere* in a shortlist of
> 20–100 candidates. (This is L85's `hybrid`.)
>
> **Stage 2 — rerank:** score those few candidates *expensively and precisely*, and reorder them so
> the best one is #1.

## Setup
Same stack as L85, plus `sentence-transformers`' **CrossEncoder** class (it ships in the same
package we already installed). Every cell has a deterministic fallback so the notebook *runs*
offline — but only the real models show the true reranking win, so **run this in Colab** for the
headline numbers.

In [ ]:
!pip install sentence-transformers rank_bm25 scikit-learn numpy -q
import re, time
import numpy as np
np.random.seed(0)
print("ready")

## §1 — Bi-encoder vs cross-encoder: why one is cheap and the other is precise

This is the whole conceptual payload of the lesson. Two architectures, two jobs.

**Bi-encoder (what dense retrieval in L85 is).** The query and each chunk are pushed through the
transformer **separately** and collapsed into one vector each. Relevance = cosine of the two
vectors. Because the chunk vectors never depend on the query, you can **embed the entire corpus
once, offline**, and at query time only embed the query and do fast dot-products. Fast and
indexable — but the query and the chunk **never attend to each other**. The model never gets to
ask "does *this* chunk answer *this* query?"; it only asks "are these two independently-made
summaries nearby?"

**Cross-encoder (the reranker).** The query and one chunk are **concatenated** and pushed through
the transformer **together**: `[CLS] query [SEP] chunk [SEP]`. Every query token can attend to
every chunk token. The output is a single relevance score for *that specific pair*. This is far
more accurate — it can tell "600 requests per minute" answers "what is the Pro rate limit" even
when a bi-encoder finds three chunks equally nearby. **The cost:** the score depends on the query,
so **nothing can be precomputed**. Scoring the whole corpus would mean one forward pass per chunk,
per query — impossibly slow. So a cross-encoder is *only ever run on a shortlist* the retriever
already narrowed down.

| | Bi-encoder (retriever) | Cross-encoder (reranker) |
|---|---|---|
| Input | query and chunk **separately** | query + chunk **together** |
| Query↔chunk attention | ❌ none | ✅ full |
| Precompute chunk reps? | ✅ embed corpus once | ❌ depends on the query |
| Cost per query | ~1 embed + N cheap dot-products | **N transformer forward passes** |
| Good at | recall over the whole corpus | precision on a small shortlist |
| Where it runs | over the entire index | over the top-20..100 only |

**The pattern that falls out of the table:** retrieve wide with the bi-encoder/hybrid (recall),
rerank narrow with the cross-encoder (precision). Neither alone is enough; together they are the
production default.

## §2 — Rebuild the L85 first stage (frozen)

Corpus, eval sets, chunker, and the three retrievers are **identical to L85** — reproduced here so
the notebook stands alone. Read past this if it is familiar; nothing changes until §3. This is our
held-constant Stage 1.

In [ ]:
DOCS = {
 "refunds": ("Nimbus Refund Policy. Customers on the monthly plan may request a full refund "
   "within 14 days of any charge. Annual plans are refundable on a prorated basis for the "
   "remaining unused months. Refunds are issued to the original payment method and take 5 to "
   "10 business days to appear. One-time setup fees are non-refundable."),
 "ratelimits": ("Nimbus API Rate Limits. The Free tier allows 60 requests per minute. The Pro "
   "tier allows 600 requests per minute. The Enterprise tier allows 6000 requests per minute. "
   "Exceeding your limit returns HTTP 429. Each 429 response includes a Retry-After header "
   "telling you how many seconds to wait before retrying."),
 "sso": ("Nimbus Single Sign-On. SSO is available on the Enterprise plan only. We support SAML "
   "2.0 and OIDC. To configure SAML, an administrator uploads the identity provider metadata XML "
   "in the Security settings page. Just-in-time user provisioning is enabled by default so new "
   "users are created on first login."),
 "retention": ("Nimbus Data Retention. Application logs are retained for 30 days. Deleted "
   "projects are held in a recoverable trash state for 90 days before permanent deletion. "
   "Customers on the Enterprise plan can configure a custom retention window of up to 7 years "
   "for compliance."),
 "security": ("Nimbus Security. All customer data is encrypted at rest using AES-256 and in "
   "transit using TLS 1.3. Nimbus is SOC 2 Type II certified. Access to production systems "
   "requires hardware security keys. We run third-party penetration tests twice per year."),
 "credentials": ("Nimbus Account Access. If you are locked out, use the credential recovery flow "
   "on the sign-in page: enter your email and we send a one-time link that lets you set a new "
   "secret. Links expire after 30 minutes. Enabling two-factor authentication is strongly "
   "recommended for all accounts."),
 "pricing": ("Nimbus Pricing. The Free tier costs nothing and includes one project. The Pro tier "
   "costs 49 dollars per month and includes ten projects. The Enterprise tier is custom-priced "
   "and includes unlimited projects, SSO, and a dedicated support manager."),
 "support": ("Nimbus Support SLAs. Free tier support is community-only. Pro tier guarantees a "
   "first response within one business day. Enterprise tier guarantees a first response within "
   "one hour for urgent issues, twenty-four hours a day, seven days a week."),
}

EVAL = [
 {"q":"how many days to get a refund on a monthly plan","gold":"refunds","expect":"14 days"},
 {"q":"what is the Pro tier rate limit","gold":"ratelimits","expect":"600 requests per minute"},
 {"q":"which plans include single sign-on","gold":"sso","expect":"Enterprise"},
 {"q":"how long are deleted projects recoverable","gold":"retention","expect":"90 days"},
 {"q":"what encryption is used for data at rest","gold":"security","expect":"AES-256"},
 {"q":"how much does the Pro plan cost per month","gold":"pricing","expect":"49 dollars"},
 {"q":"how fast does Enterprise support respond to urgent issues","gold":"support","expect":"one hour"},
 {"q":"what is the maximum custom retention window for compliance","gold":"retention","expect":"7 years"},
]

PARAPHRASE = [
 {"q":"I forgot how to log in, how do I reset my password","gold":"credentials"},
 {"q":"my app keeps getting blocked after too many calls","gold":"ratelimits"},
 {"q":"can I get my money back","gold":"refunds"},
 {"q":"how do you keep my information safe from hackers","gold":"security"},
 {"q":"log in with my company account","gold":"sso"},
 {"q":"when will you delete my stuff for good","gold":"retention"},
]
print(f"{len(DOCS)} docs, {len(EVAL)} lexical-friendly Qs, {len(PARAPHRASE)} paraphrase Qs.")

In [ ]:
# L84's winning chunker, frozen
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]
def _recurse(text, size, seps):
    if len(text) <= size: return [text]
    if not seps or seps[0] == "":
        return [text[i:i+size] for i in range(0, len(text), size)]
    sep = seps[0]; chunks, cur = [], ""
    for p in text.split(sep):
        cand = p if cur == "" else cur + sep + p
        if len(cand) <= size:
            cur = cand
        else:
            if cur: chunks.append(cur)
            if len(p) > size:
                chunks.extend(_recurse(p, size, seps[1:])); cur = ""
            else:
                cur = p
    if cur: chunks.append(cur)
    return chunks
def chunk_recursive(doc_id, text, size=180):
    out = []
    for piece in _recurse(text, size, SEPARATORS):
        piece = piece.strip()
        if piece: out.append({"id": f"{doc_id}#{len(out)}", "doc": doc_id, "text": piece})
    return out

CHUNKS = []
for d, t in DOCS.items():
    CHUNKS.extend(chunk_recursive(d, t, size=180))
TEXTS = [c["text"] for c in CHUNKS]
print(f"{len(CHUNKS)} chunks (frozen).")

In [ ]:
# Three retrievers from L85, self-contained, with the same offline fallback for dense.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
def tok(s): return re.findall(r"[a-z0-9]+", s.lower())

class TfidfRetriever:
    name="tfidf"
    def __init__(self, chunks):
        self.chunks=chunks; self.vec=TfidfVectorizer(stop_words="english")
        self.M=self.vec.fit_transform([c["text"] for c in chunks])
    def search(self, q, k=5):
        s=cosine_similarity(self.vec.transform([q]), self.M)[0]; order=np.argsort(-s)[:k]
        return [(self.chunks[i], float(s[i])) for i in order]

class BM25Retriever:
    name="bm25"
    def __init__(self, chunks):
        self.chunks=chunks; self.bm25=BM25Okapi([tok(c["text"]) for c in chunks])
    def search(self, q, k=5):
        s=np.array(self.bm25.get_scores(tok(q))); order=np.argsort(-s)[:k]
        return [(self.chunks[i], float(s[i])) for i in order]

MODEL_LOADED=False
try:
    from sentence_transformers import SentenceTransformer
    _emb=SentenceTransformer("all-MiniLM-L6-v2")
    def embed(t): return np.asarray(_emb.encode(list(t), normalize_embeddings=True), dtype="float32")
    MODEL_LOADED=True
    print("Dense backend: REAL all-MiniLM-L6-v2.")
except Exception as e:
    from sklearn.feature_extraction.text import TfidfVectorizer as _TV
    _fb=_TV(analyzer="char_wb", ngram_range=(3,5)); _fb.fit(TEXTS)
    def embed(t):
        X=_fb.transform(list(t)).toarray().astype("float32")
        n=np.linalg.norm(X,axis=1,keepdims=True); n[n==0]=1.0; return X/n
    print("Dense backend: FALLBACK char-ngram (", type(e).__name__, "). Run in Colab for real.")

class DenseRetriever:
    name="dense"
    def __init__(self, chunks):
        self.chunks=chunks; self.E=embed([c["text"] for c in chunks])
    def search(self, q, k=5):
        s=self.E @ embed([q])[0]; order=np.argsort(-s)[:k]
        return [(self.chunks[i], float(s[i])) for i in order]

tfidf=TfidfRetriever(CHUNKS); bm25=BM25Retriever(CHUNKS); dense=DenseRetriever(CHUNKS)
print("Stage-1 retrievers ready.")

In [ ]:
# L85's hybrid: RRF-fuse bm25 + dense. This is our FROZEN Stage 1.
def rrf_fuse(rankings, k=60, topn=5):
    fused={}
    for ranked_ids in rankings:
        for rank, cid in enumerate(ranked_ids, start=1):
            fused[cid]=fused.get(cid,0.0)+1.0/(k+rank)
    return sorted(fused.items(), key=lambda kv:-kv[1])[:topn]

class HybridRetriever:
    name="hybrid"
    def __init__(self, retrievers, k=60, pool=10):
        self.retrievers=retrievers; self.k=k; self.pool=pool
        self.by_id={c["id"]:c for c in retrievers[0].chunks}
    def search(self, q, k=5):
        rankings=[[c["id"] for c,_ in r.search(q, self.pool)] for r in self.retrievers]
        fused=rrf_fuse(rankings, k=self.k, topn=k)
        return [(self.by_id[cid], sc) for cid, sc in fused]

hybrid=HybridRetriever([bm25, dense], pool=12)
print("Frozen Stage 1 = hybrid(bm25, dense).")

## §3 — See FM3 with your own eyes: hybrid gets it in the shortlist, not at #1

Before we fix anything, let's *measure the disease*. For each query we ask hybrid for a shortlist of
`k=8` and record **the rank at which the gold document first appears**. If reranking is worth
building, we should see plenty of golds sitting at rank 2, 3, 5 — retrieved, but not on top.

In [ ]:
def gold_rank(results, gold):
    for i,(c,_) in enumerate(results, start=1):
        if c["doc"]==gold: return i
    return None

print("Where does hybrid rank the gold chunk?  (shortlist k=8)\n")
print(f"{'query':52}{'gold@rank':>10}")
print("-"*62)
not_first=0
for e in EVAL+PARAPHRASE:
    res=hybrid.search(e["q"], k=8)
    r=gold_rank(res, e["gold"])
    flag="" if r==1 else "   <-- not #1"
    if r!=1: not_first+=1
    print(f"{e['q'][:50]:52}{str(r):>10}{flag}")
print("-"*62)
print(f"\n{not_first}/{len(EVAL+PARAPHRASE)} queries have the gold chunk retrieved but NOT ranked #1.")
print("Those are exactly the answers FM3 loses if the generator only reads the top 1-3 chunks.")
print("(Fallback backend understates this; the real models in Colab show the honest picture.)")

## §4 — The cross-encoder reranker

`sentence-transformers` ships a `CrossEncoder` wrapper. We load
`cross-encoder/ms-marco-MiniLM-L-6-v2` — a small model fine-tuned on the MS-MARCO passage-ranking
task, which is the canonical off-the-shelf reranker. Its `.predict([(query, chunk), ...])` returns
one relevance score per pair.

As always there is a deterministic fallback so the notebook runs offline. The fallback is an
honest *joint* scorer — it reads the query and chunk **together** (token overlap + exact-phrase
bonus), which is the defining property of a cross-encoder, even though it is not a neural model. It
will improve the lexical `EVAL` ordering; only the real model improves the `PARAPHRASE` ordering.

In [ ]:
RERANK_LOADED=False
try:
    from sentence_transformers import CrossEncoder
    _ce=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    def ce_scores(query, chunks):
        pairs=[(query, c) for c in chunks]
        return np.asarray(_ce.predict(pairs), dtype="float32")
    RERANK_LOADED=True
    print("Reranker: REAL cross-encoder/ms-marco-MiniLM-L-6-v2 (neural, query<->chunk attention).")
except Exception as e:
    def ce_scores(query, chunks):
        # Joint lexical stand-in: reads (query, chunk) TOGETHER. Not neural; no semantic match.
        qset=set(tok(query)); out=[]
        for c in chunks:
            ct=tok(c); cset=set(ct)
            overlap=len(qset & cset)/(len(qset) or 1)          # recall of query tokens
            phrase=1.0 if query.lower() in c.lower() else 0.0  # exact phrase bonus
            dens=len(qset & cset)/(len(ct) or 1)               # term density
            out.append(overlap + 0.5*phrase + 0.25*dens)
        return np.asarray(out, dtype="float32")
    print("Reranker: FALLBACK joint-lexical (", type(e).__name__, "). Run in Colab for the real win.")

class Reranker:
    """Stage 2: reorder a shortlist of (chunk, score) by cross-encoder relevance."""
    name="reranker"
    def __init__(self, score_fn): self.score_fn=score_fn
    def rerank(self, query, candidates, k=5):
        chunks=[c for c,_ in candidates]
        s=self.score_fn(query, [c["text"] for c in chunks])
        order=np.argsort(-s)
        return [(chunks[i], float(s[i])) for i in order[:k]]

reranker=Reranker(ce_scores)

# Sanity: one query, show the reorder
demo_q="what is the Pro tier rate limit"
short=hybrid.search(demo_q, k=8)
print("\nQUERY:", demo_q)
print("Stage-1 (hybrid) order:  ", [c["id"] for c,_ in short])
print("Stage-2 (reranked) order:", [c["id"] for c,_ in reranker.rerank(demo_q, short, k=8)])

## §5 — The retrieve-then-rerank pipeline

Now compose the two stages behind the **same `search(q, k)` interface** every retriever in this
course has used since L83. That is the point: the pipeline is a drop-in replacement — nothing
downstream (the generator, the eval harness, the FastAPI route) has to change.

- **`retrieve_pool`** — how wide Stage 1 casts (the reranker's recall ceiling; it can only reorder
  what it's given).
- **`k`** — how many survive to the generator.

In [ ]:
class RerankPipeline:
    name="hybrid+rerank"
    def __init__(self, retriever, reranker, retrieve_pool=12):
        self.retriever=retriever; self.reranker=reranker; self.retrieve_pool=retrieve_pool
    def search(self, q, k=5):
        shortlist=self.retriever.search(q, k=self.retrieve_pool)   # wide + cheap
        return self.reranker.rerank(q, shortlist, k=k)             # narrow + precise

pipeline=RerankPipeline(hybrid, reranker, retrieve_pool=12)

# Same three problem queries from §3, before vs after
print(f"{'query':50}{'hybrid@1':>10}{'reranked@1':>12}")
print("-"*72)
for e in EVAL+PARAPHRASE:
    h=hybrid.search(e["q"], k=8); p=pipeline.search(e["q"], k=8)
    hr=gold_rank(h, e["gold"]); pr=gold_rank(p, e["gold"])
    print(f"{e['q'][:48]:50}{str(hr):>10}{str(pr):>12}")
print("-"*72)
print("Reranking pulls golds that were retrieved-but-buried up toward rank 1.")

## §6 — The scoreboard: does reranking actually lift the metrics?

Same `evaluate` harness as L85 (`hit@1` = fraction where the #1 chunk is gold; `MRR` = mean of
1/rank of the gold). We compare the frozen Stage-1 retrievers against the reranked pipeline on
**both** eval sets. Reranking should raise `hit@1` and `MRR` without us having touched the index,
the chunker, or the embeddings.

In [ ]:
def evaluate(retriever, evalset, k=8):
    hit1=0.0; mrr=0.0
    for e in evalset:
        res=retriever.search(e["q"], k=k)
        if res and res[0][0]["doc"]==e["gold"]: hit1+=1
        r=gold_rank(res, e["gold"]); mrr+=(1.0/r) if r else 0.0
    n=len(evalset); return hit1/n, mrr/n

ROWS=[dense, hybrid, pipeline]
print(f"{'system':16}{'EVAL hit@1':>12}{'EVAL MRR':>10}{'PARA hit@1':>12}{'PARA MRR':>10}")
print("-"*60)
BOARD={}
for r in ROWS:
    eh,em=evaluate(r, EVAL); ph,pm=evaluate(r, PARAPHRASE)
    BOARD[r.name]={"eval_hit1":eh,"eval_mrr":em,"para_hit1":ph,"para_mrr":pm}
    print(f"{r.name:16}{eh:>12.2f}{em:>10.2f}{ph:>12.2f}{pm:>10.2f}")
print("-"*60)
if RERANK_LOADED and MODEL_LOADED:
    print("Real models: hybrid+rerank should match or beat hybrid on hit@1 AND MRR on both evals.")
else:
    print("(Fallback backends understate PARA. The lexical EVAL column still shows the reorder.)")

## §7 — The bill: cross-encoders cost one forward pass per candidate

Reranking is not free, and the cost is **linear in the pool size**. If Stage 1 hands the reranker
`retrieve_pool=P` candidates, that is **P transformer forward passes per query** — on top of
retrieval. This is the dial you tune:

- **Small P (e.g. 10–20):** cheap, low latency, but if the gold sits at rank 30 in Stage 1 the
  reranker never sees it — you are capped by Stage-1 recall.
- **Large P (e.g. 100):** higher recall ceiling, but latency and GPU cost climb linearly, and you
  are paying to rerank a lot of obvious junk.

The sweep below shows the trade: `hit@1`/`MRR` vs the pool size, and the forward-pass count that
sets the bill. In production you pick the smallest P where the metric curve has flattened.

In [ ]:
def eval_pool(P, evalset, k=5):
    pipe=RerankPipeline(HybridRetriever([bm25, dense], pool=P), reranker, retrieve_pool=P)
    eh,em=evaluate(pipe, evalset, k=k)
    return eh, em

print(f"{'pool P':>7}{'fwd passes/q':>14}{'EVAL hit@1':>12}{'EVAL MRR':>10}{'PARA hit@1':>12}{'PARA MRR':>10}")
print("-"*67)
for P in [4, 8, 12, 20, len(CHUNKS)]:
    eh,em=eval_pool(P, EVAL); ph,pm=eval_pool(P, PARAPHRASE)
    tag=" (=full corpus)" if P==len(CHUNKS) else ""
    print(f"{P:>7}{P:>14}{eh:>12.2f}{em:>10.2f}{ph:>12.2f}{pm:>10.2f}{tag}")
print("-"*67)
print("Note P=full-corpus turns the reranker into a (slow) exhaustive scorer: best possible")
print("quality, worst possible cost. The retriever exists precisely so you don't do that.")

# A crude latency read (real cross-encoder only)
if RERANK_LOADED:
    q=EVAL[0]["q"]; cand=[c["text"] for c,_ in hybrid.search(q, k=20)]
    t0=time.time(); _=ce_scores(q, cand); dt=(time.time()-t0)*1000
    print(f"\nReal cross-encoder: {len(cand)} pairs scored in {dt:.0f} ms "
          f"(~{dt/len(cand):.1f} ms/candidate). Latency scales with P.")

## §8 — Sidebar: the LLM *is* a reranker (listwise / RankGPT)

A cross-encoder scores each `(query, chunk)` pair **independently** — that is *pointwise*
reranking. You can instead hand an LLM the whole shortlist at once and ask it to **order the list**
— *listwise* reranking (the "RankGPT" pattern). The LLM sees all candidates together, so it can
reason about relative relevance and redundancy in a way pointwise scoring cannot.

Trade-offs, honestly:

- ✅ Highest quality; understands intent, negation, multi-hop relevance; no fine-tuned model to host.
- ❌ Far more expensive and slower than a small cross-encoder (a full LLM call per query, L22 costs).
- ❌ Needs care: output-order bias, must return *all* ids, can hallucinate an id not in the list.

Rule of thumb: **cross-encoder for the hot path, LLM reranker only when quality justifies the
bill** — or as the *judge* that tells you whether a cheaper reranker is good enough (that is L88).
The cell runs only if an Anthropic key is present; otherwise it skips cleanly.

In [ ]:
# Optional: listwise LLM reranker. Skips gracefully with no key (e.g. offline scheduled run).
import os, json as _json
def llm_rerank(query, candidates, k=5):
    try:
        from anthropic import Anthropic
        key=os.environ.get("ANTHROPIC_API_KEY")
        try:
            from google.colab import userdata
            key=key or userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            pass
        if not key:
            print("No ANTHROPIC_API_KEY -> skipping LLM reranker (this is fine)."); return None
        client=Anthropic(api_key=key)
        chunks=[c for c,_ in candidates]
        listing="\n".join(f"[{i}] {c['text']}" for i,c in enumerate(chunks))
        prompt=(f"Query: {query}\n\nPassages:\n{listing}\n\n"
                f"Return ONLY a JSON list of passage indices from MOST to LEAST relevant, "
                f"e.g. [3,0,1]. Include every index exactly once.")
        msg=client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=200,
                                    messages=[{"role":"user","content":prompt}])
        txt=msg.content[0].text
        order=_json.loads(re.search(r"\[[\d,\s]*\]", txt).group(0))
        seen=set(); order=[i for i in order if isinstance(i,int) and 0<=i<len(chunks) and not (i in seen or seen.add(i))]
        order+=[i for i in range(len(chunks)) if i not in seen]   # repair missing ids
        return [(chunks[i], 0.0) for i in order[:k]]
    except Exception as e:
        print("LLM reranker unavailable:", type(e).__name__, "-> skipping."); return None

_q=PARAPHRASE[0]["q"]
_res=llm_rerank(_q, hybrid.search(_q, k=8), k=5)
if _res:
    print("QUERY:", _q)
    print("LLM-reranked top-1:", _res[0][0]["id"], "->", _res[0][0]["text"][:70])

## §9 — What to actually ship

A production reranking stage, distilled:

- **Default architecture:** `hybrid (BM25+dense) → cross-encoder rerank → top-k to the LLM`. This is
  the workhorse; reach for it before anything fancier.
- **Model:** start with `cross-encoder/ms-marco-MiniLM-L-6-v2` (small, fast, strong). Upgrade to a
  larger MiniLM/`bge-reranker` only if L88 evals say you need it. A hosted rerank API (Cohere Rerank,
  Voyage rerank) is a fine no-GPU option — same two-stage shape, someone else's model.
- **Pool size:** tune `retrieve_pool` empirically (the §7 sweep). Typical: retrieve 50–100, rerank
  to 3–8. Pick the smallest pool where quality plateaus.
- **When to skip reranking:** tiny corpora where hybrid already hits ceiling; ultra-low-latency
  paths where the extra forward passes blow the budget; or when L88 shows no measurable lift.
- **Always measure.** Reranking usually helps, but not always on *your* corpus. Ship it behind the
  eval harness (L88), not on faith.

## §10 — Ten reranking pitfalls

1. **Reranking with too small a pool.** The reranker can only reorder what Stage 1 returned. If the
   gold sits at rank 40 and you retrieve 10, no reranker can save you — you are recall-capped.
2. **Reranking the whole corpus.** Cross-encoders cost one forward pass per candidate. Running one
   over every chunk defeats the entire point of having a retriever; it is quadratically slow.
3. **Confusing the two encoders.** A bi-encoder (dense retriever) and a cross-encoder (reranker) are
   different models with different jobs. You cannot index with a cross-encoder or rerank precisely
   with a bi-encoder.
4. **Latency blindness.** Reranking adds P forward passes to every query. Measure p95 (L73) — a
   reranker that doubles tail latency may violate your SLO even if hit@1 improves.
5. **No eval gate.** "Reranking is best practice" is not evidence it helps *your* data. Gate it on
   the L88 metrics; sometimes hybrid alone is already at ceiling.
6. **Score-scale confusion.** Cross-encoder scores are not probabilities or cosines; they are
   uncalibrated logits. Use them to *order*, not as confidence, and never compare across models.
7. **Truncation.** Cross-encoders have a max input length; a query + a long chunk can overflow and
   silently truncate the chunk's tail — where the answer might be. Keep chunks within the model's
   window (ties back to L84 chunk sizing).
8. **Domain mismatch.** MS-MARCO rerankers are trained on web Q&A. On legal, medical, or code
   corpora an off-the-shelf reranker can *underperform* — evaluate before trusting, consider a
   domain-tuned reranker.
9. **LLM-reranker fragility.** Listwise LLM reranking can drop ids, invent ids, or anchor on list
   order. Always validate the returned permutation and repair missing ids (as the §8 cell does).
10. **Reranking a bad retriever.** Reranking is a *precision* tool, not a *recall* tool. If Stage 1
    has poor recall (bad chunking L84, or wrong retriever L85), fix that first — a reranker cannot
    retrieve what was never fetched.

## §11 — Verification

Machine-checkable claims from this lesson. On the real Colab backend all should pass; the
fallback backend passes the structural checks but not the semantic-lift ones (noted inline).

In [ ]:
checks=[]
def check(name, cond): checks.append((name, bool(cond))); print(("PASS" if cond else "FAIL"), "-", name)

# 1. Pipeline preserves the search(q,k) interface and returns k results
res=pipeline.search("what is the Pro tier rate limit", k=5)
check("pipeline.search returns a list of (chunk, score)", isinstance(res,list) and len(res)==5 and len(res[0])==2)

# 2. Reranking never loses a chunk that Stage 1 returned (it only reorders a subset)
q=EVAL[1]["q"]; pool=hybrid.search(q, k=8); pool_ids={c["id"] for c,_ in pool}
rr=reranker.rerank(q, pool, k=8); rr_ids={c["id"] for c,_ in rr}
check("reranker output is a subset of the Stage-1 shortlist", rr_ids <= pool_ids)

# 3. Cost is linear: reranking pool P issues exactly P pair-scores
import numpy as _np
probe_chunks=[c["text"] for c,_ in hybrid.search(q, k=12)]
check("ce_scores returns one score per candidate (O(P) cost)", len(ce_scores(q, probe_chunks))==len(probe_chunks))

# 4. Reranking does not reduce quality vs Stage 1 (hit@1) on the lexical EVAL set
h_hit,_=evaluate(hybrid, EVAL); p_hit,_=evaluate(pipeline, EVAL)
check("hybrid+rerank hit@1 >= hybrid hit@1 on EVAL", p_hit >= h_hit - 1e-9)

# 5. Semantic-lift check (real models only)
if RERANK_LOADED and MODEL_LOADED:
    _,h_mrr=evaluate(hybrid, PARAPHRASE); _,p_mrr=evaluate(pipeline, PARAPHRASE)
    check("real reranker: MRR on PARA >= hybrid MRR", p_mrr >= h_mrr - 1e-9)
else:
    print("SKIP - semantic-lift check needs real models (run in Colab).")

print(f"\n{sum(1 for _,ok in checks if ok)}/{len(checks)} structural checks passed.")

## Summary

- Retrievers score chunks **independently and cheaply**, so their *ordering* is coarse: the right
  chunk lands in the top-k but often not at #1. When the generator reads only the top few, that is
  **FM3** — a retrieved-but-buried answer, lost.
- A **bi-encoder** embeds query and chunk separately (indexable, fast, whole-corpus). A
  **cross-encoder** reads `(query, chunk)` together with full attention (precise, but one forward
  pass per pair — only runnable on a shortlist).
- The production pattern is **two stages**: retrieve wide and cheap (L85 hybrid), then **rerank
  narrow and precise** (cross-encoder). Same `search(q, k)` interface — a drop-in upgrade.
- Cost is **linear in the pool size**: tune `retrieve_pool` to the smallest value where quality
  plateaus. Reranking the whole corpus defeats the retriever's purpose.
- An **LLM can rerank listwise** (RankGPT) for top quality at higher cost — or serve as the judge
  that decides whether a cheap reranker is good enough (L88).
- **Always eval-gate it (L88).** Reranking usually helps; prove it does on *your* corpus.

### Homework
1. **Pool-vs-quality curve.** Extend the §7 sweep with `P = 6, 10, 16, 24`. Plot `hit@1` vs `P`.
   Where does the curve flatten on *this* corpus? That flat point is your production `retrieve_pool`.
2. **Swap the reranker.** Try `cross-encoder/ms-marco-MiniLM-L-12-v2` (bigger) and, if you can, a
   `BAAI/bge-reranker-base`. Re-run §6. Does the larger model earn its extra latency here?
3. **Pointwise vs listwise.** With an API key, compare the §4 cross-encoder against the §8 LLM
   reranker on `PARAPHRASE`. Measure the MRR gap *and* the latency/cost gap. Is listwise worth it?
4. **Break it on purpose.** Set `retrieve_pool=3` and re-run §6. Show that reranking cannot recover
   golds Stage 1 didn't fetch — the recall-cap pitfall (#1) in numbers.
5. **Wire it into your project.** Replace the L85 `HybridRetriever` in your RAG pipeline with this
   `RerankPipeline`, keeping `search(q, k)`. Re-run the L83 baseline end-to-end and record the lift.

### Up next — L87: Grounding & citations
We now retrieve the right chunk *and* rank it #1. But the generator can still ignore it, blend it
with training-data memory, or answer with no traceable source — **FM5: hallucination / no
provenance.** L87 makes the model answer *only* from the retrieved evidence and attach inline
citations, so every claim points back to a chunk id — and a claim with no support is refused, not
invented.